**Azure Databricks Final Assessment**

**Domain: Hospital Operations Analytics**

Objective

Build an end-to-end Databricks pipeline using:

PySpark DataFrames

Spark SQL

Delta Lake

CRUD

MERGE / UPSERT

History

Time Travel

VACUUM

Parquet to Delta

Incremental Load

DLT

Unity Catalog

Governance


Part 0 — Prepare Bigger Dataset

Students can run this first in a normal Python notebook.

In [1]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Hospital Operations Analytics").getOrCreate()

In [2]:
patients_data = [
(1001,"Aarav Khan","Hyderabad",29,"Male","Gold","2023-01-10"),
(1002,"Priya Reddy","Bengaluru",34,"Female","Silver","2023-01-15"),
(1003,"Rahul Mehta","Mumbai",41,"Male","Gold","2023-02-02"),
(1004,"Sneha Kapoor","Delhi",26,"Female","Bronze","2023-02-18"),
(1005,"Kiran Patel","Ahmedabad",37,"Male","Silver","2023-03-01"),
(1006,"Ananya Das","Kolkata",31,"Female","Gold","2023-03-12"),
(1007,"Vikram Singh","Chennai",45,"Male","Bronze","2023-03-20"),
(1008,"Meera Nair","Kochi",28,"Female","Silver","2023-04-05"),
(1009,"Farhan Ali","Hyderabad",39,"Male","Gold","2023-04-15"),
(1010,"Divya Menon","Bengaluru",33,"Female","Silver","2023-04-21"),
(1011,"Arjun Iyer","Chennai",52,"Male","Gold","2023-05-01"),
(1012,"Neha Gupta","Delhi",25,"Female","Bronze","2023-05-11"),
(1013,"Sanjay Rao","Mumbai",48,"Male","Silver","2023-05-19"),
(1014,"Kavya Sharma","Hyderabad",30,"Female","Gold","2023-06-01"),
(1015,"Nikhil Verma","Pune",36,"Male","Silver","2023-06-12"),
(1016,"Ayesha Khan","Kolkata",27,"Female","Bronze","2023-06-20"),
(1017,"Manish Yadav","Lucknow",44,"Male","Gold","2023-07-05"),
(1018,"Pooja Shah","Ahmedabad",32,"Female","Silver","2023-07-18"),
(1019,"Rohan Nair","Kochi",40,"Male","Gold","2023-08-01"),
(1020,"Lakshmi Rao","Chennai",35,"Female","Silver","2023-08-14")
]

patients_columns = [
"patient_id","patient_name","city","age","gender","membership","registration_date"
]

patients_df = spark.createDataFrame(patients_data, patients_columns)

In [3]:
doctors_data = [
(201,"Dr Sameer Sharma","Cardiology","Hyderabad",1200),
(202,"Dr Kavita Iyer","Dermatology","Bengaluru",800),
(203,"Dr Imran Khan","Orthopedics","Mumbai",1500),
(204,"Dr Ramesh Reddy","Pediatrics","Delhi",900),
(205,"Dr Anita Mehta","Neurology","Hyderabad",2000),
(206,"Dr Joseph Mathew","Cardiology","Chennai",1300),
(207,"Dr Fatima Ali","Dermatology","Kolkata",850),
(208,"Dr Arvind Rao","Orthopedics","Bengaluru",1400),
(209,"Dr Leela Nair","Neurology","Kochi",1900),
(210,"Dr Ganesh Patil","General Medicine","Pune",700)
]

doctors_columns = [
"doctor_id","doctor_name","specialization","doctor_city","consultation_fee"
]

doctors_df = spark.createDataFrame(doctors_data, doctors_columns)

In [4]:
visits_data = [
(1,1001,201,"2024-03-01","Completed",2),
(2,1002,202,"2024-03-01","Completed",1),
(3,1003,203,"2024-03-02","Completed",3),
(4,1004,204,"2024-03-02","Pending",1),
(5,1005,206,"2024-03-03","Completed",2),
(6,1006,205,"2024-03-03","Completed",4),
(7,1007,207,"2024-03-04","Cancelled",1),
(8,1008,208,"2024-03-04","Completed",2),
(9,1009,201,"2024-03-05","Completed",1),
(10,1010,202,"2024-03-05","Completed",2),
(11,1011,205,"2024-03-06","Pending",3),
(12,1012,204,"2024-03-06","Completed",1),
(13,1013,203,"2024-03-07","Completed",2),
(14,1014,201,"2024-03-07","Completed",3),
(15,1015,210,"2024-03-08","Completed",1),
(16,1016,207,"2024-03-08","Cancelled",2),
(17,1017,209,"2024-03-09","Completed",4),
(18,1018,206,"2024-03-09","Completed",2),
(19,1019,209,"2024-03-10","Completed",3),
(20,1020,206,"2024-03-10","Pending",2),
(21,1001,205,"2024-03-11","Completed",3),
(22,1003,208,"2024-03-11","Completed",2),
(23,1006,201,"2024-03-12","Completed",1),
(24,1009,210,"2024-03-12","Completed",2),
(25,1014,202,"2024-03-13","Completed",1)
]

visits_columns = [
"visit_id","patient_id","doctor_id","visit_date","visit_status","tests_count"
]

visits_df = spark.createDataFrame(visits_data, visits_columns)

In [5]:
payments_data = [
(301,1,5200,"UPI","Paid"),
(302,2,2800,"Credit Card","Paid"),
(303,3,7500,"Cash","Paid"),
(304,4,2900,"UPI","Pending"),
(305,5,5300,"Debit Card","Paid"),
(306,6,10000,"Credit Card","Paid"),
(307,7,2850,"Cash","Cancelled"),
(308,8,5400,"UPI","Paid"),
(309,9,3200,"UPI","Paid"),
(310,10,4800,"Credit Card","Paid"),
(311,11,8000,"UPI","Pending"),
(312,12,2900,"Cash","Paid"),
(313,13,5500,"Credit Card","Paid"),
(314,14,7200,"UPI","Paid"),
(315,15,2700,"Debit Card","Paid"),
(316,16,4850,"Cash","Cancelled"),
(317,17,9900,"Credit Card","Paid"),
(318,18,5300,"UPI","Paid"),
(319,19,7900,"Debit Card","Paid"),
(320,20,5300,"UPI","Pending"),
(321,21,8000,"UPI","Paid"),
(322,22,5400,"Credit Card","Paid"),
(323,23,3200,"Cash","Paid"),
(324,24,4700,"UPI","Paid"),
(325,25,2800,"UPI","Paid")
]

payments_columns = [
"payment_id","visit_id","bill_amount","payment_mode","payment_status"
]

payments_df = spark.createDataFrame(payments_data, payments_columns)

Assessment Tasks

Part 1 — DataFrame Fundamentals

1. Display all four DataFrames.


In [6]:
patients_df.show()
doctors_df.show()
visits_df.show()
payments_df.show()

+----------+------------+---------+---+------+----------+-----------------+
|patient_id|patient_name|     city|age|gender|membership|registration_date|
+----------+------------+---------+---+------+----------+-----------------+
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|
|      1002| Priya Reddy|Bengaluru| 34|Female|    Silver|       2023-01-15|
|      1003| Rahul Mehta|   Mumbai| 41|  Male|      Gold|       2023-02-02|
|      1004|Sneha Kapoor|    Delhi| 26|Female|    Bronze|       2023-02-18|
|      1005| Kiran Patel|Ahmedabad| 37|  Male|    Silver|       2023-03-01|
|      1006|  Ananya Das|  Kolkata| 31|Female|      Gold|       2023-03-12|
|      1007|Vikram Singh|  Chennai| 45|  Male|    Bronze|       2023-03-20|
|      1008|  Meera Nair|    Kochi| 28|Female|    Silver|       2023-04-05|
|      1009|  Farhan Ali|Hyderabad| 39|  Male|      Gold|       2023-04-15|
|      1010| Divya Menon|Bengaluru| 33|Female|    Silver|       2023-04-21|
|      1011|

2. Print schema for each DataFrame.

In [7]:
patients_df.printSchema()
doctors_df.printSchema()
visits_df.printSchema()
payments_df.printSchema()

root
 |-- patient_id: long (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- membership: string (nullable = true)
 |-- registration_date: string (nullable = true)

root
 |-- doctor_id: long (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- specialization: string (nullable = true)
 |-- doctor_city: string (nullable = true)
 |-- consultation_fee: long (nullable = true)

root
 |-- visit_id: long (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- doctor_id: long (nullable = true)
 |-- visit_date: string (nullable = true)
 |-- visit_status: string (nullable = true)
 |-- tests_count: long (nullable = true)

root
 |-- payment_id: long (nullable = true)
 |-- visit_id: long (nullable = true)
 |-- bill_amount: long (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)



3. Count records in each DataFrame.

In [8]:
patients_df.count()
doctors_df.count()
visits_df.count()
payments_df.count()

25

4. Display first 10 patient records.

In [9]:
patients_df.show(10)

+----------+------------+---------+---+------+----------+-----------------+
|patient_id|patient_name|     city|age|gender|membership|registration_date|
+----------+------------+---------+---+------+----------+-----------------+
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|
|      1002| Priya Reddy|Bengaluru| 34|Female|    Silver|       2023-01-15|
|      1003| Rahul Mehta|   Mumbai| 41|  Male|      Gold|       2023-02-02|
|      1004|Sneha Kapoor|    Delhi| 26|Female|    Bronze|       2023-02-18|
|      1005| Kiran Patel|Ahmedabad| 37|  Male|    Silver|       2023-03-01|
|      1006|  Ananya Das|  Kolkata| 31|Female|      Gold|       2023-03-12|
|      1007|Vikram Singh|  Chennai| 45|  Male|    Bronze|       2023-03-20|
|      1008|  Meera Nair|    Kochi| 28|Female|    Silver|       2023-04-05|
|      1009|  Farhan Ali|Hyderabad| 39|  Male|      Gold|       2023-04-15|
|      1010| Divya Menon|Bengaluru| 33|Female|    Silver|       2023-04-21|
+----------+

5. Display only patient name, city, membership, and age.

In [10]:
patients_df.select("patient_name","city","membership","age").show()

+------------+---------+----------+---+
|patient_name|     city|membership|age|
+------------+---------+----------+---+
|  Aarav Khan|Hyderabad|      Gold| 29|
| Priya Reddy|Bengaluru|    Silver| 34|
| Rahul Mehta|   Mumbai|      Gold| 41|
|Sneha Kapoor|    Delhi|    Bronze| 26|
| Kiran Patel|Ahmedabad|    Silver| 37|
|  Ananya Das|  Kolkata|      Gold| 31|
|Vikram Singh|  Chennai|    Bronze| 45|
|  Meera Nair|    Kochi|    Silver| 28|
|  Farhan Ali|Hyderabad|      Gold| 39|
| Divya Menon|Bengaluru|    Silver| 33|
|  Arjun Iyer|  Chennai|      Gold| 52|
|  Neha Gupta|    Delhi|    Bronze| 25|
|  Sanjay Rao|   Mumbai|    Silver| 48|
|Kavya Sharma|Hyderabad|      Gold| 30|
|Nikhil Verma|     Pune|    Silver| 36|
| Ayesha Khan|  Kolkata|    Bronze| 27|
|Manish Yadav|  Lucknow|      Gold| 44|
|  Pooja Shah|Ahmedabad|    Silver| 32|
|  Rohan Nair|    Kochi|      Gold| 40|
| Lakshmi Rao|  Chennai|    Silver| 35|
+------------+---------+----------+---+



6. Display only doctor name, specialization, and consultation fee.

In [11]:
doctors_df.select("doctor_name","specialization","consultation_fee").show()

+----------------+----------------+----------------+
|     doctor_name|  specialization|consultation_fee|
+----------------+----------------+----------------+
|Dr Sameer Sharma|      Cardiology|            1200|
|  Dr Kavita Iyer|     Dermatology|             800|
|   Dr Imran Khan|     Orthopedics|            1500|
| Dr Ramesh Reddy|      Pediatrics|             900|
|  Dr Anita Mehta|       Neurology|            2000|
|Dr Joseph Mathew|      Cardiology|            1300|
|   Dr Fatima Ali|     Dermatology|             850|
|   Dr Arvind Rao|     Orthopedics|            1400|
|   Dr Leela Nair|       Neurology|            1900|
| Dr Ganesh Patil|General Medicine|             700|
+----------------+----------------+----------------+



7. Display all visits with status Completed.


In [12]:
visits_df.filter(visits_df.visit_status=="Completed").show()

+--------+----------+---------+----------+------------+-----------+
|visit_id|patient_id|doctor_id|visit_date|visit_status|tests_count|
+--------+----------+---------+----------+------------+-----------+
|       1|      1001|      201|2024-03-01|   Completed|          2|
|       2|      1002|      202|2024-03-01|   Completed|          1|
|       3|      1003|      203|2024-03-02|   Completed|          3|
|       5|      1005|      206|2024-03-03|   Completed|          2|
|       6|      1006|      205|2024-03-03|   Completed|          4|
|       8|      1008|      208|2024-03-04|   Completed|          2|
|       9|      1009|      201|2024-03-05|   Completed|          1|
|      10|      1010|      202|2024-03-05|   Completed|          2|
|      12|      1012|      204|2024-03-06|   Completed|          1|
|      13|      1013|      203|2024-03-07|   Completed|          2|
|      14|      1014|      201|2024-03-07|   Completed|          3|
|      15|      1015|      210|2024-03-08|   Com

8. Display all pending visits.

In [13]:
visits_df.filter(visits_df.visit_status=="Pending").show()

+--------+----------+---------+----------+------------+-----------+
|visit_id|patient_id|doctor_id|visit_date|visit_status|tests_count|
+--------+----------+---------+----------+------------+-----------+
|       4|      1004|      204|2024-03-02|     Pending|          1|
|      11|      1011|      205|2024-03-06|     Pending|          3|
|      20|      1020|      206|2024-03-10|     Pending|          2|
+--------+----------+---------+----------+------------+-----------+



9. Display patients from Hyderabad and Bengaluru.

In [14]:
from pyspark.sql.functions import col
patients_df.filter(col("city").isin("Hyderabad","Bengaluru")).show()

+----------+------------+---------+---+------+----------+-----------------+
|patient_id|patient_name|     city|age|gender|membership|registration_date|
+----------+------------+---------+---+------+----------+-----------------+
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|
|      1002| Priya Reddy|Bengaluru| 34|Female|    Silver|       2023-01-15|
|      1009|  Farhan Ali|Hyderabad| 39|  Male|      Gold|       2023-04-15|
|      1010| Divya Menon|Bengaluru| 33|Female|    Silver|       2023-04-21|
|      1014|Kavya Sharma|Hyderabad| 30|Female|      Gold|       2023-06-01|
+----------+------------+---------+---+------+----------+-----------------+



10. Display doctors whose consultation fee is greater than 1000.

In [15]:
doctors_df.filter(col("consultation_fee") > 1000).show()

+---------+----------------+--------------+-----------+----------------+
|doctor_id|     doctor_name|specialization|doctor_city|consultation_fee|
+---------+----------------+--------------+-----------+----------------+
|      201|Dr Sameer Sharma|    Cardiology|  Hyderabad|            1200|
|      203|   Dr Imran Khan|   Orthopedics|     Mumbai|            1500|
|      205|  Dr Anita Mehta|     Neurology|  Hyderabad|            2000|
|      206|Dr Joseph Mathew|    Cardiology|    Chennai|            1300|
|      208|   Dr Arvind Rao|   Orthopedics|  Bengaluru|            1400|
|      209|   Dr Leela Nair|     Neurology|      Kochi|            1900|
+---------+----------------+--------------+-----------+----------------+



Part 2 — DataFrame Transformations

11. Convert registration_date into date type.


In [16]:
from pyspark.sql.functions import *
patients_df = patients_df.withColumn(
    "registration_date",
    to_date(col("registration_date"))
)

patients_df.show()

+----------+------------+---------+---+------+----------+-----------------+
|patient_id|patient_name|     city|age|gender|membership|registration_date|
+----------+------------+---------+---+------+----------+-----------------+
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|
|      1002| Priya Reddy|Bengaluru| 34|Female|    Silver|       2023-01-15|
|      1003| Rahul Mehta|   Mumbai| 41|  Male|      Gold|       2023-02-02|
|      1004|Sneha Kapoor|    Delhi| 26|Female|    Bronze|       2023-02-18|
|      1005| Kiran Patel|Ahmedabad| 37|  Male|    Silver|       2023-03-01|
|      1006|  Ananya Das|  Kolkata| 31|Female|      Gold|       2023-03-12|
|      1007|Vikram Singh|  Chennai| 45|  Male|    Bronze|       2023-03-20|
|      1008|  Meera Nair|    Kochi| 28|Female|    Silver|       2023-04-05|
|      1009|  Farhan Ali|Hyderabad| 39|  Male|      Gold|       2023-04-15|
|      1010| Divya Menon|Bengaluru| 33|Female|    Silver|       2023-04-21|
|      1011|

12. Convert visit_date into date type.

In [17]:
visits_df=visits_df.withColumn(
    "visit_date",
    to_date(col("visit_date"))
)
visits_df.show()

+--------+----------+---------+----------+------------+-----------+
|visit_id|patient_id|doctor_id|visit_date|visit_status|tests_count|
+--------+----------+---------+----------+------------+-----------+
|       1|      1001|      201|2024-03-01|   Completed|          2|
|       2|      1002|      202|2024-03-01|   Completed|          1|
|       3|      1003|      203|2024-03-02|   Completed|          3|
|       4|      1004|      204|2024-03-02|     Pending|          1|
|       5|      1005|      206|2024-03-03|   Completed|          2|
|       6|      1006|      205|2024-03-03|   Completed|          4|
|       7|      1007|      207|2024-03-04|   Cancelled|          1|
|       8|      1008|      208|2024-03-04|   Completed|          2|
|       9|      1009|      201|2024-03-05|   Completed|          1|
|      10|      1010|      202|2024-03-05|   Completed|          2|
|      11|      1011|      205|2024-03-06|     Pending|          3|
|      12|      1012|      204|2024-03-06|   Com

13. Add a column test_cost .

In [18]:
visits_df.withColumn(
    "test_cost",
    col("tests_count") * 500
).show()

+--------+----------+---------+----------+------------+-----------+---------+
|visit_id|patient_id|doctor_id|visit_date|visit_status|tests_count|test_cost|
+--------+----------+---------+----------+------------+-----------+---------+
|       1|      1001|      201|2024-03-01|   Completed|          2|     1000|
|       2|      1002|      202|2024-03-01|   Completed|          1|      500|
|       3|      1003|      203|2024-03-02|   Completed|          3|     1500|
|       4|      1004|      204|2024-03-02|     Pending|          1|      500|
|       5|      1005|      206|2024-03-03|   Completed|          2|     1000|
|       6|      1006|      205|2024-03-03|   Completed|          4|     2000|
|       7|      1007|      207|2024-03-04|   Cancelled|          1|      500|
|       8|      1008|      208|2024-03-04|   Completed|          2|     1000|
|       9|      1009|      201|2024-03-05|   Completed|          1|      500|
|      10|      1010|      202|2024-03-05|   Completed|         

14. Add a column estimated_total_bill .

In [19]:
visits_join_df = visits_df.join(
    doctors_df,
    on="doctor_id",
    how="inner"
)

visits_join_df = visits_join_df.withColumn(
    "estimated_total_bill",
    col("consultation_fee") + (col("tests_count") * 500)
)

visits_join_df.show()

+---------+--------+----------+----------+------------+-----------+----------------+--------------+-----------+----------------+--------------------+
|doctor_id|visit_id|patient_id|visit_date|visit_status|tests_count|     doctor_name|specialization|doctor_city|consultation_fee|estimated_total_bill|
+---------+--------+----------+----------+------------+-----------+----------------+--------------+-----------+----------------+--------------------+
|      201|       1|      1001|2024-03-01|   Completed|          2|Dr Sameer Sharma|    Cardiology|  Hyderabad|            1200|                2200|
|      201|       9|      1009|2024-03-05|   Completed|          1|Dr Sameer Sharma|    Cardiology|  Hyderabad|            1200|                1700|
|      201|      14|      1014|2024-03-07|   Completed|          3|Dr Sameer Sharma|    Cardiology|  Hyderabad|            1200|                2700|
|      201|      23|      1006|2024-03-12|   Completed|          1|Dr Sameer Sharma|    Cardiology| 

15. Create age_category .

In [20]:
type(patients_df)

pyspark.sql.classic.dataframe.DataFrame

In [21]:
patients_df = patients_df.withColumn(
    "age_category",
    when(col("age") < 30, "Young")
    .when(col("age") < 45, "Adult")
    .otherwise("Senior")
)

patients_df.show()

+----------+------------+---------+---+------+----------+-----------------+------------+
|patient_id|patient_name|     city|age|gender|membership|registration_date|age_category|
+----------+------------+---------+---+------+----------+-----------------+------------+
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|       Young|
|      1002| Priya Reddy|Bengaluru| 34|Female|    Silver|       2023-01-15|       Adult|
|      1003| Rahul Mehta|   Mumbai| 41|  Male|      Gold|       2023-02-02|       Adult|
|      1004|Sneha Kapoor|    Delhi| 26|Female|    Bronze|       2023-02-18|       Young|
|      1005| Kiran Patel|Ahmedabad| 37|  Male|    Silver|       2023-03-01|       Adult|
|      1006|  Ananya Das|  Kolkata| 31|Female|      Gold|       2023-03-12|       Adult|
|      1007|Vikram Singh|  Chennai| 45|  Male|    Bronze|       2023-03-20|      Senior|
|      1008|  Meera Nair|    Kochi| 28|Female|    Silver|       2023-04-05|       Young|
|      1009|  Farhan 


16. Create membership_priority .

In [22]:
patients_df = patients_df.withColumn(
    "membership_priority",
    when(col("membership")=="Gold",1)
    .when(col("membership")=="Silver",2)
    .otherwise(3)
)

patients_df.show()

+----------+------------+---------+---+------+----------+-----------------+------------+-------------------+
|patient_id|patient_name|     city|age|gender|membership|registration_date|age_category|membership_priority|
+----------+------------+---------+---+------+----------+-----------------+------------+-------------------+
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|       Young|                  1|
|      1002| Priya Reddy|Bengaluru| 34|Female|    Silver|       2023-01-15|       Adult|                  2|
|      1003| Rahul Mehta|   Mumbai| 41|  Male|      Gold|       2023-02-02|       Adult|                  1|
|      1004|Sneha Kapoor|    Delhi| 26|Female|    Bronze|       2023-02-18|       Young|                  3|
|      1005| Kiran Patel|Ahmedabad| 37|  Male|    Silver|       2023-03-01|       Adult|                  2|
|      1006|  Ananya Das|  Kolkata| 31|Female|      Gold|       2023-03-12|       Adult|                  1|
|      1007|Vikram 

17. Create visit_priority .

In [23]:
visits_df = visits_df.withColumn(
    "visit_priority",
    when(col("visit_status")=="Pending","High")
    .otherwise("Normal")
)
visits_df.show()

+--------+----------+---------+----------+------------+-----------+--------------+
|visit_id|patient_id|doctor_id|visit_date|visit_status|tests_count|visit_priority|
+--------+----------+---------+----------+------------+-----------+--------------+
|       1|      1001|      201|2024-03-01|   Completed|          2|        Normal|
|       2|      1002|      202|2024-03-01|   Completed|          1|        Normal|
|       3|      1003|      203|2024-03-02|   Completed|          3|        Normal|
|       4|      1004|      204|2024-03-02|     Pending|          1|          High|
|       5|      1005|      206|2024-03-03|   Completed|          2|        Normal|
|       6|      1006|      205|2024-03-03|   Completed|          4|        Normal|
|       7|      1007|      207|2024-03-04|   Cancelled|          1|        Normal|
|       8|      1008|      208|2024-03-04|   Completed|          2|        Normal|
|       9|      1009|      201|2024-03-05|   Completed|          1|        Normal|
|   

18. Create high_bill_flag .

In [24]:
payments_df = payments_df.withColumn(
    "high_bill_flag",
    when(col("bill_amount") > 7000,"YES")
    .otherwise("NO")
)
payments_df.show()

+----------+--------+-----------+------------+--------------+--------------+
|payment_id|visit_id|bill_amount|payment_mode|payment_status|high_bill_flag|
+----------+--------+-----------+------------+--------------+--------------+
|       301|       1|       5200|         UPI|          Paid|            NO|
|       302|       2|       2800| Credit Card|          Paid|            NO|
|       303|       3|       7500|        Cash|          Paid|           YES|
|       304|       4|       2900|         UPI|       Pending|            NO|
|       305|       5|       5300|  Debit Card|          Paid|            NO|
|       306|       6|      10000| Credit Card|          Paid|           YES|
|       307|       7|       2850|        Cash|     Cancelled|            NO|
|       308|       8|       5400|         UPI|          Paid|            NO|
|       309|       9|       3200|         UPI|          Paid|            NO|
|       310|      10|       4800| Credit Card|          Paid|            NO|

19. Rename patient_name to full_name .

In [25]:
patients_df = patients_df.withColumnRenamed(
    "patient_name",
    "full_name"
)

20. Drop any temporary column created during transformation.

In [26]:
visits_df = visits_df.drop("test_cost")

Part 3 — Joins

21. Join patients with visits.













In [27]:
patient_visit_df = patients_df.join(
    visits_df,
    "patient_id"
)
patient_visit_df.show()

+----------+------------+---------+---+------+----------+-----------------+------------+-------------------+--------+---------+----------+------------+-----------+--------------+
|patient_id|   full_name|     city|age|gender|membership|registration_date|age_category|membership_priority|visit_id|doctor_id|visit_date|visit_status|tests_count|visit_priority|
+----------+------------+---------+---+------+----------+-----------------+------------+-------------------+--------+---------+----------+------------+-----------+--------------+
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|       Young|                  1|       1|      201|2024-03-01|   Completed|          2|        Normal|
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|       Young|                  1|      21|      205|2024-03-11|   Completed|          3|        Normal|
|      1002| Priya Reddy|Bengaluru| 34|Female|    Silver|       2023-01-15|       Adult|                 

22. Join visits with doctors.

In [28]:
visit_doctor_df = visits_df.join(
    doctors_df,
    "doctor_id"
)
visit_doctor_df.show()

+---------+--------+----------+----------+------------+-----------+--------------+----------------+--------------+-----------+----------------+
|doctor_id|visit_id|patient_id|visit_date|visit_status|tests_count|visit_priority|     doctor_name|specialization|doctor_city|consultation_fee|
+---------+--------+----------+----------+------------+-----------+--------------+----------------+--------------+-----------+----------------+
|      201|       1|      1001|2024-03-01|   Completed|          2|        Normal|Dr Sameer Sharma|    Cardiology|  Hyderabad|            1200|
|      201|       9|      1009|2024-03-05|   Completed|          1|        Normal|Dr Sameer Sharma|    Cardiology|  Hyderabad|            1200|
|      201|      14|      1014|2024-03-07|   Completed|          3|        Normal|Dr Sameer Sharma|    Cardiology|  Hyderabad|            1200|
|      201|      23|      1006|2024-03-12|   Completed|          1|        Normal|Dr Sameer Sharma|    Cardiology|  Hyderabad|          

23. Join visits with payments.

In [29]:
visit_payment_df = visits_df.join(
    payments_df,
    "visit_id"
)
visit_payment_df.show()

+--------+----------+---------+----------+------------+-----------+--------------+----------+-----------+------------+--------------+--------------+
|visit_id|patient_id|doctor_id|visit_date|visit_status|tests_count|visit_priority|payment_id|bill_amount|payment_mode|payment_status|high_bill_flag|
+--------+----------+---------+----------+------------+-----------+--------------+----------+-----------+------------+--------------+--------------+
|       1|      1001|      201|2024-03-01|   Completed|          2|        Normal|       301|       5200|         UPI|          Paid|            NO|
|       2|      1002|      202|2024-03-01|   Completed|          1|        Normal|       302|       2800| Credit Card|          Paid|            NO|
|       3|      1003|      203|2024-03-02|   Completed|          3|        Normal|       303|       7500|        Cash|          Paid|           YES|
|       4|      1004|      204|2024-03-02|     Pending|          1|          High|       304|       2900| 

24. Create a final joined DataFrame containing patient, doctor, visit, and payment details.

In [30]:
final_df = patients_df.join(
    visits_df,
    "patient_id"
).join(
    doctors_df,
    "doctor_id"
).join(
    payments_df,
    "visit_id"
)

25. Show patient name, city, doctor name, specialization, visit status, and bill amount.

In [31]:
final_df.select(
    "full_name",
    "city",
    "doctor_name",
    "specialization",
    "visit_status",
    "bill_amount"
).show()

+------------+---------+----------------+----------------+------------+-----------+
|   full_name|     city|     doctor_name|  specialization|visit_status|bill_amount|
+------------+---------+----------------+----------------+------------+-----------+
|  Aarav Khan|Hyderabad|Dr Sameer Sharma|      Cardiology|   Completed|       5200|
| Priya Reddy|Bengaluru|  Dr Kavita Iyer|     Dermatology|   Completed|       2800|
| Rahul Mehta|   Mumbai|   Dr Imran Khan|     Orthopedics|   Completed|       7500|
|Sneha Kapoor|    Delhi| Dr Ramesh Reddy|      Pediatrics|     Pending|       2900|
| Kiran Patel|Ahmedabad|Dr Joseph Mathew|      Cardiology|   Completed|       5300|
|  Ananya Das|  Kolkata|  Dr Anita Mehta|       Neurology|   Completed|      10000|
|Vikram Singh|  Chennai|   Dr Fatima Ali|     Dermatology|   Cancelled|       2850|
|  Meera Nair|    Kochi|   Dr Arvind Rao|     Orthopedics|   Completed|       5400|
|  Farhan Ali|Hyderabad|Dr Sameer Sharma|      Cardiology|   Completed|     

26. Find visits where patient city and doctor city are different.

In [32]:
final_df.filter(
    col("city") != col("doctor_city")
).show()


+--------+---------+----------+------------+---------+---+------+----------+-----------------+------------+-------------------+----------+------------+-----------+--------------+----------------+----------------+-----------+----------------+----------+-----------+------------+--------------+--------------+
|visit_id|doctor_id|patient_id|   full_name|     city|age|gender|membership|registration_date|age_category|membership_priority|visit_date|visit_status|tests_count|visit_priority|     doctor_name|  specialization|doctor_city|consultation_fee|payment_id|bill_amount|payment_mode|payment_status|high_bill_flag|
+--------+---------+----------+------------+---------+---+------+----------+-----------------+------------+-------------------+----------+------------+-----------+--------------+----------------+----------------+-----------+----------------+----------+-----------+------------+--------------+--------------+
|       5|      206|      1005| Kiran Patel|Ahmedabad| 37|  Male|    Silver|

27. Find completed visits with paid payments.

In [33]:
final_df.filter(
    (col("visit_status")=="Completed") &
    (col("payment_status")=="Paid")
).show()


+--------+---------+----------+------------+---------+---+------+----------+-----------------+------------+-------------------+----------+------------+-----------+--------------+----------------+----------------+-----------+----------------+----------+-----------+------------+--------------+--------------+
|visit_id|doctor_id|patient_id|   full_name|     city|age|gender|membership|registration_date|age_category|membership_priority|visit_date|visit_status|tests_count|visit_priority|     doctor_name|  specialization|doctor_city|consultation_fee|payment_id|bill_amount|payment_mode|payment_status|high_bill_flag|
+--------+---------+----------+------------+---------+---+------+----------+-----------------+------------+-------------------+----------+------------+-----------+--------------+----------------+----------------+-----------+----------------+----------+-----------+------------+--------------+--------------+
|       1|      201|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|

28. Find pending visits with pending payments.

In [34]:
final_df.filter(
    (col("visit_status")=="Pending") &
    (col("payment_status")=="Pending")
).show()

+--------+---------+----------+------------+-------+---+------+----------+-----------------+------------+-------------------+----------+------------+-----------+--------------+----------------+--------------+-----------+----------------+----------+-----------+------------+--------------+--------------+
|visit_id|doctor_id|patient_id|   full_name|   city|age|gender|membership|registration_date|age_category|membership_priority|visit_date|visit_status|tests_count|visit_priority|     doctor_name|specialization|doctor_city|consultation_fee|payment_id|bill_amount|payment_mode|payment_status|high_bill_flag|
+--------+---------+----------+------------+-------+---+------+----------+-----------------+------------+-------------------+----------+------------+-----------+--------------+----------------+--------------+-----------+----------------+----------+-----------+------------+--------------+--------------+
|       4|      204|      1004|Sneha Kapoor|  Delhi| 26|Female|    Bronze|       2023-02

29. Find cancelled visits with cancelled payments.

In [35]:
final_df.filter(
    (col("visit_status")=="Cancelled") &
    (col("payment_status")=="Cancelled")
).show()

+--------+---------+----------+------------+-------+---+------+----------+-----------------+------------+-------------------+----------+------------+-----------+--------------+-------------+--------------+-----------+----------------+----------+-----------+------------+--------------+--------------+
|visit_id|doctor_id|patient_id|   full_name|   city|age|gender|membership|registration_date|age_category|membership_priority|visit_date|visit_status|tests_count|visit_priority|  doctor_name|specialization|doctor_city|consultation_fee|payment_id|bill_amount|payment_mode|payment_status|high_bill_flag|
+--------+---------+----------+------------+-------+---+------+----------+-----------------+------------+-------------------+----------+------------+-----------+--------------+-------------+--------------+-----------+----------------+----------+-----------+------------+--------------+--------------+
|       7|      207|      1007|Vikram Singh|Chennai| 45|  Male|    Bronze|       2023-03-20|     

30. Find patients who visited more than once.

In [36]:
final_df.groupBy(
    "patient_id","full_name"
).count().filter(
    col("count") > 1
).show()

+----------+------------+-----+
|patient_id|   full_name|count|
+----------+------------+-----+
|      1003| Rahul Mehta|    2|
|      1014|Kavya Sharma|    2|
|      1006|  Ananya Das|    2|
|      1001|  Aarav Khan|    2|
|      1009|  Farhan Ali|    2|
+----------+------------+-----+



Part 4 — Aggregations


31. Count patients by city.

In [37]:
patients_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|    Kochi|    2|
|  Chennai|    3|
|   Mumbai|    2|
|Ahmedabad|    2|
|  Kolkata|    2|
|    Delhi|    2|
|Bengaluru|    2|
|Hyderabad|    3|
|  Lucknow|    1|
|     Pune|    1|
+---------+-----+



32. Count patients by membership.

In [38]:
patients_df.groupBy("membership").count().show()

+----------+-----+
|membership|count|
+----------+-----+
|    Silver|    8|
|      Gold|    8|
|    Bronze|    4|
+----------+-----+



33. Count doctors by specialization.

In [39]:
doctors_df.groupBy("specialization").count().show()

+----------------+-----+
|  specialization|count|
+----------------+-----+
|       Neurology|    2|
|     Dermatology|    2|
|      Cardiology|    2|
|      Pediatrics|    1|
|     Orthopedics|    2|
|General Medicine|    1|
+----------------+-----+



34. Count visits by status.

In [40]:
visits_df.groupBy("visit_status").count().show()

+------------+-----+
|visit_status|count|
+------------+-----+
|   Completed|   20|
|   Cancelled|    2|
|     Pending|    3|
+------------+-----+



35. Count payments by payment mode.

In [41]:
payments_df.groupBy("payment_mode").count().show()

+------------+-----+
|payment_mode|count|
+------------+-----+
| Credit Card|    6|
|        Cash|    5|
|  Debit Card|    3|
|         UPI|   11|
+------------+-----+



36. Calculate total bill amount.

In [42]:
payments_df.agg(
    sum("bill_amount").alias("total_bill")
).show()

+----------+
|total_bill|
+----------+
|    133600|
+----------+



37. Calculate average bill amount.

In [43]:
payments_df.agg(
    avg("bill_amount").alias("average_bill")
).show()

+------------+
|average_bill|
+------------+
|      5344.0|
+------------+



38. Calculate total revenue by city.

In [44]:
final_df.groupBy("city").agg(
    sum("bill_amount").alias("revenue")
).show()

+---------+-------+
|     city|revenue|
+---------+-------+
|    Kochi|  13300|
|  Chennai|  16150|
|  Lucknow|   9900|
|   Mumbai|  18400|
|Ahmedabad|  10600|
|  Kolkata|  18050|
|     Pune|   2700|
|    Delhi|   5800|
|Bengaluru|   7600|
|Hyderabad|  31100|
+---------+-------+



39. Calculate total revenue by specialization.

In [45]:
final_df.groupBy("specialization").agg(
    sum("bill_amount").alias("revenue")
).show()

+----------------+-------+
|  specialization|revenue|
+----------------+-------+
|       Neurology|  43800|
|     Dermatology|  18100|
|      Cardiology|  34700|
|      Pediatrics|   5800|
|     Orthopedics|  23800|
|General Medicine|   7400|
+----------------+-------+



40. Calculate total revenue by doctor.

In [46]:
final_df.groupBy("doctor_name").agg(
    sum("bill_amount").alias("revenue")
).show()

+----------------+-------+
|     doctor_name|revenue|
+----------------+-------+
|   Dr Fatima Ali|   7700|
|Dr Joseph Mathew|  15900|
|  Dr Kavita Iyer|  10400|
|   Dr Arvind Rao|  10800|
| Dr Ganesh Patil|   7400|
|   Dr Leela Nair|  17800|
|   Dr Imran Khan|  13000|
|Dr Sameer Sharma|  18800|
|  Dr Anita Mehta|  26000|
| Dr Ramesh Reddy|   5800|
+----------------+-------+



Part 5 — Spark SQL

41. Create temp views for all DataFrames.



In [47]:
patients_df.createOrReplaceTempView("patients")
doctors_df.createOrReplaceTempView("doctors")
visits_df.createOrReplaceTempView("visits")
payments_df.createOrReplaceTempView("payments")

42. Show all patients using SQL.

In [48]:
spark.sql("SELECT * FROM patients").show()

+----------+------------+---------+---+------+----------+-----------------+------------+-------------------+
|patient_id|   full_name|     city|age|gender|membership|registration_date|age_category|membership_priority|
+----------+------------+---------+---+------+----------+-----------------+------------+-------------------+
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|       Young|                  1|
|      1002| Priya Reddy|Bengaluru| 34|Female|    Silver|       2023-01-15|       Adult|                  2|
|      1003| Rahul Mehta|   Mumbai| 41|  Male|      Gold|       2023-02-02|       Adult|                  1|
|      1004|Sneha Kapoor|    Delhi| 26|Female|    Bronze|       2023-02-18|       Young|                  3|
|      1005| Kiran Patel|Ahmedabad| 37|  Male|    Silver|       2023-03-01|       Adult|                  2|
|      1006|  Ananya Das|  Kolkata| 31|Female|      Gold|       2023-03-12|       Adult|                  1|
|      1007|Vikram 

43. Find Cardiology visits using SQL.

In [49]:
spark.sql("""
SELECT *
FROM visits v
JOIN doctors d
ON v.doctor_id = d.doctor_id
WHERE specialization='Cardiology'
""").show()

+--------+----------+---------+----------+------------+-----------+--------------+---------+----------------+--------------+-----------+----------------+
|visit_id|patient_id|doctor_id|visit_date|visit_status|tests_count|visit_priority|doctor_id|     doctor_name|specialization|doctor_city|consultation_fee|
+--------+----------+---------+----------+------------+-----------+--------------+---------+----------------+--------------+-----------+----------------+
|       1|      1001|      201|2024-03-01|   Completed|          2|        Normal|      201|Dr Sameer Sharma|    Cardiology|  Hyderabad|            1200|
|       9|      1009|      201|2024-03-05|   Completed|          1|        Normal|      201|Dr Sameer Sharma|    Cardiology|  Hyderabad|            1200|
|      14|      1014|      201|2024-03-07|   Completed|          3|        Normal|      201|Dr Sameer Sharma|    Cardiology|  Hyderabad|            1200|
|      23|      1006|      201|2024-03-12|   Completed|          1|        N

44. Find revenue by city using SQL.

In [50]:
spark.sql("""
SELECT p.city,
SUM(pay.bill_amount) AS revenue
FROM patients p
JOIN visits v
ON p.patient_id=v.patient_id
JOIN payments pay
ON v.visit_id=pay.visit_id
GROUP BY p.city
""").show()

+---------+-------+
|     city|revenue|
+---------+-------+
|    Kochi|  13300|
|  Chennai|  16150|
|  Lucknow|   9900|
|   Mumbai|  18400|
|Ahmedabad|  10600|
|  Kolkata|  18050|
|     Pune|   2700|
|    Delhi|   5800|
|Bengaluru|   7600|
|Hyderabad|  31100|
+---------+-------+



45. Find revenue by specialization using SQL.

In [51]:
spark.sql("""
SELECT d.specialization,
SUM(pay.bill_amount) AS revenue
FROM doctors d
JOIN visits v
ON d.doctor_id=v.doctor_id
JOIN payments pay
ON v.visit_id=pay.visit_id
GROUP BY d.specialization
""").show()


+----------------+-------+
|  specialization|revenue|
+----------------+-------+
|       Neurology|  43800|
|     Dermatology|  18100|
|      Cardiology|  34700|
|      Pediatrics|   5800|
|     Orthopedics|  23800|
|General Medicine|   7400|
+----------------+-------+



46. Find top 5 highest bill visits.

In [52]:
spark.sql("""
SELECT *
FROM payments
ORDER BY bill_amount DESC
LIMIT 5
""").show()

+----------+--------+-----------+------------+--------------+--------------+
|payment_id|visit_id|bill_amount|payment_mode|payment_status|high_bill_flag|
+----------+--------+-----------+------------+--------------+--------------+
|       306|       6|      10000| Credit Card|          Paid|           YES|
|       317|      17|       9900| Credit Card|          Paid|           YES|
|       311|      11|       8000|         UPI|       Pending|           YES|
|       321|      21|       8000|         UPI|          Paid|           YES|
|       319|      19|       7900|  Debit Card|          Paid|           YES|
+----------+--------+-----------+------------+--------------+--------------+



47. Count visits per doctor.

In [53]:
spark.sql("""
SELECT doctor_id,
COUNT(*) AS total_visits
FROM visits
GROUP BY doctor_id
""").show()

+---------+------------+
|doctor_id|total_visits|
+---------+------------+
|      202|           3|
|      201|           4|
|      203|           2|
|      208|           2|
|      207|           2|
|      205|           3|
|      206|           3|
|      204|           2|
|      209|           2|
|      210|           2|
+---------+------------+



48. Count visits per patient.

In [54]:
spark.sql("""
SELECT patient_id,
COUNT(*) AS total_visits
FROM visits
GROUP BY patient_id
""").show()

+----------+------------+
|patient_id|total_visits|
+----------+------------+
|      1010|           1|
|      1002|           1|
|      1012|           1|
|      1009|           2|
|      1007|           1|
|      1011|           1|
|      1005|           1|
|      1001|           2|
|      1008|           1|
|      1004|           1|
|      1006|           2|
|      1003|           2|
|      1016|           1|
|      1013|           1|
|      1018|           1|
|      1020|           1|
|      1015|           1|
|      1014|           2|
|      1019|           1|
|      1017|           1|
+----------+------------+



49. Find average bill by payment mode.

In [55]:
spark.sql("""
SELECT payment_mode,
AVG(bill_amount) AS avg_bill
FROM payments
GROUP BY payment_mode
""").show()

+------------+-----------------+
|payment_mode|         avg_bill|
+------------+-----------------+
| Credit Card|           6400.0|
|        Cash|           4260.0|
|  Debit Card|           5300.0|
|         UPI|5272.727272727273|
+------------+-----------------+



50. Find patients with total billing above 10000.

In [56]:
spark.sql("""
SELECT p.full_name,
SUM(pay.bill_amount) AS total_bill
FROM patients p
JOIN visits v
ON p.patient_id=v.patient_id
JOIN payments pay
ON v.visit_id=pay.visit_id
GROUP BY p.full_name
HAVING SUM(pay.bill_amount) > 10000
""").show()

+-----------+----------+
|  full_name|total_bill|
+-----------+----------+
| Aarav Khan|     13200|
| Ananya Das|     13200|
|Rahul Mehta|     12900|
+-----------+----------+



Part 6 — Window Functions

51. Rank patients by total bill within city.


In [57]:
from pyspark.sql.window import Window

In [58]:
final_df.withColumn(
    "rank",
    rank().over(
        Window.partitionBy("city")
        .orderBy(desc("bill_amount"))
    )
).show()

+--------+---------+----------+------------+---------+---+------+----------+-----------------+------------+-------------------+----------+------------+-----------+--------------+----------------+----------------+-----------+----------------+----------+-----------+------------+--------------+--------------+----+
|visit_id|doctor_id|patient_id|   full_name|     city|age|gender|membership|registration_date|age_category|membership_priority|visit_date|visit_status|tests_count|visit_priority|     doctor_name|  specialization|doctor_city|consultation_fee|payment_id|bill_amount|payment_mode|payment_status|high_bill_flag|rank|
+--------+---------+----------+------------+---------+---+------+----------+-----------------+------------+-------------------+----------+------------+-----------+--------------+----------------+----------------+-----------+----------------+----------+-----------+------------+--------------+--------------+----+
|       5|      206|      1005| Kiran Patel|Ahmedabad| 37|  M

52. Rank doctors by total revenue within specialization.

In [59]:
final_df.groupBy(
    "specialization","doctor_name"
).agg(
    sum("bill_amount").alias("revenue")
).withColumn(
    "rank",
    rank().over(
        Window.partitionBy("specialization")
        .orderBy(desc("revenue"))
    )
).show()

+----------------+----------------+-------+----+
|  specialization|     doctor_name|revenue|rank|
+----------------+----------------+-------+----+
|      Cardiology|Dr Sameer Sharma|  18800|   1|
|      Cardiology|Dr Joseph Mathew|  15900|   2|
|     Dermatology|  Dr Kavita Iyer|  10400|   1|
|     Dermatology|   Dr Fatima Ali|   7700|   2|
|General Medicine| Dr Ganesh Patil|   7400|   1|
|       Neurology|  Dr Anita Mehta|  26000|   1|
|       Neurology|   Dr Leela Nair|  17800|   2|
|     Orthopedics|   Dr Imran Khan|  13000|   1|
|     Orthopedics|   Dr Arvind Rao|  10800|   2|
|      Pediatrics| Dr Ramesh Reddy|   5800|   1|
+----------------+----------------+-------+----+



53. Use ROW_NUMBER to find top billing patient per city.

In [60]:
final_df.withColumn(
    "row_num",
    row_number().over(
        Window.partitionBy("city")
        .orderBy(desc("bill_amount"))
    )
).filter(
    col("row_num")==1
).show()


+--------+---------+----------+------------+---------+---+------+----------+-----------------+------------+-------------------+----------+------------+-----------+--------------+----------------+----------------+-----------+----------------+----------+-----------+------------+--------------+--------------+-------+
|visit_id|doctor_id|patient_id|   full_name|     city|age|gender|membership|registration_date|age_category|membership_priority|visit_date|visit_status|tests_count|visit_priority|     doctor_name|  specialization|doctor_city|consultation_fee|payment_id|bill_amount|payment_mode|payment_status|high_bill_flag|row_num|
+--------+---------+----------+------------+---------+---+------+----------+-----------------+------------+-------------------+----------+------------+-----------+--------------+----------------+----------------+-----------+----------------+----------+-----------+------------+--------------+--------------+-------+
|       5|      206|      1005| Kiran Patel|Ahmedaba

54. Use DENSE_RANK to rank doctors by consultation fee within specialization.

In [61]:
doctors_df.withColumn(
    "dense_rank",
    dense_rank().over(
        Window.partitionBy("specialization")
        .orderBy(desc("consultation_fee"))
    )
).show()

+---------+----------------+----------------+-----------+----------------+----------+
|doctor_id|     doctor_name|  specialization|doctor_city|consultation_fee|dense_rank|
+---------+----------------+----------------+-----------+----------------+----------+
|      206|Dr Joseph Mathew|      Cardiology|    Chennai|            1300|         1|
|      201|Dr Sameer Sharma|      Cardiology|  Hyderabad|            1200|         2|
|      207|   Dr Fatima Ali|     Dermatology|    Kolkata|             850|         1|
|      202|  Dr Kavita Iyer|     Dermatology|  Bengaluru|             800|         2|
|      210| Dr Ganesh Patil|General Medicine|       Pune|             700|         1|
|      205|  Dr Anita Mehta|       Neurology|  Hyderabad|            2000|         1|
|      209|   Dr Leela Nair|       Neurology|      Kochi|            1900|         2|
|      203|   Dr Imran Khan|     Orthopedics|     Mumbai|            1500|         1|
|      208|   Dr Arvind Rao|     Orthopedics|  Bengalu

55. Find top 2 doctors by revenue.

In [62]:
final_df.groupBy(
    "doctor_name"
).agg(
    sum("bill_amount").alias("revenue")
).orderBy(
    desc("revenue")
).show(2)

+----------------+-------+
|     doctor_name|revenue|
+----------------+-------+
|  Dr Anita Mehta|  26000|
|Dr Sameer Sharma|  18800|
+----------------+-------+
only showing top 2 rows


56. Create running total revenue by visit date.

In [63]:
final_df.groupBy(
    "visit_date"
).agg(
    sum("bill_amount").alias("daily_revenue")
).withColumn(
    "running_total",
    sum("daily_revenue").over(
        Window.orderBy("visit_date")
    )
).show()

+----------+-------------+-------------+
|visit_date|daily_revenue|running_total|
+----------+-------------+-------------+
|2024-03-01|         8000|         8000|
|2024-03-02|        10400|        18400|
|2024-03-03|        15300|        33700|
|2024-03-04|         8250|        41950|
|2024-03-05|         8000|        49950|
|2024-03-06|        10900|        60850|
|2024-03-07|        12700|        73550|
|2024-03-08|         7550|        81100|
|2024-03-09|        15200|        96300|
|2024-03-10|        13200|       109500|
|2024-03-11|        13400|       122900|
|2024-03-12|         7900|       130800|
|2024-03-13|         2800|       133600|
+----------+-------------+-------------+



57. Create running total revenue by doctor.

In [64]:
final_df.withColumn(
    "running_total",
    sum("bill_amount").over(
        Window.partitionBy("doctor_name")
        .orderBy("visit_date")
    )
).show()

+--------+---------+----------+------------+---------+---+------+----------+-----------------+------------+-------------------+----------+------------+-----------+--------------+----------------+----------------+-----------+----------------+----------+-----------+------------+--------------+--------------+-------------+
|visit_id|doctor_id|patient_id|   full_name|     city|age|gender|membership|registration_date|age_category|membership_priority|visit_date|visit_status|tests_count|visit_priority|     doctor_name|  specialization|doctor_city|consultation_fee|payment_id|bill_amount|payment_mode|payment_status|high_bill_flag|running_total|
+--------+---------+----------+------------+---------+---+------+----------+-----------------+------------+-------------------+----------+------------+-----------+--------------+----------------+----------------+-----------+----------------+----------+-----------+------------+--------------+--------------+-------------+
|       6|      205|      1006|  A

58. Rank cities by total revenue.

In [65]:
final_df.groupBy(
    "city"
).agg(
    sum("bill_amount").alias("revenue")
).withColumn(
    "rank",
    rank().over(
        Window.orderBy(desc("revenue"))
    )
).show()

+---------+-------+----+
|     city|revenue|rank|
+---------+-------+----+
|Hyderabad|  31100|   1|
|   Mumbai|  18400|   2|
|  Kolkata|  18050|   3|
|  Chennai|  16150|   4|
|    Kochi|  13300|   5|
|Ahmedabad|  10600|   6|
|  Lucknow|   9900|   7|
|Bengaluru|   7600|   8|
|    Delhi|   5800|   9|
|     Pune|   2700|  10|
+---------+-------+----+



59. Rank specializations by total revenue.

In [66]:
final_df.groupBy(
    "specialization"
).agg(
    sum("bill_amount").alias("revenue")
).withColumn(
    "rank",
    rank().over(
        Window.orderBy(desc("revenue"))
    )
).show()

+----------------+-------+----+
|  specialization|revenue|rank|
+----------------+-------+----+
|       Neurology|  43800|   1|
|      Cardiology|  34700|   2|
|     Orthopedics|  23800|   3|
|     Dermatology|  18100|   4|
|General Medicine|   7400|   5|
|      Pediatrics|   5800|   6|
+----------------+-------+----+



60. Find highest bill visit per payment mode.

In [67]:
payments_df.withColumn(
    "row_num",
    row_number().over(
        Window.partitionBy("payment_mode")
        .orderBy(desc("bill_amount"))
    )
).filter(
    col("row_num")==1
).show()

+----------+--------+-----------+------------+--------------+--------------+-------+
|payment_id|visit_id|bill_amount|payment_mode|payment_status|high_bill_flag|row_num|
+----------+--------+-----------+------------+--------------+--------------+-------+
|       303|       3|       7500|        Cash|          Paid|           YES|      1|
|       306|       6|      10000| Credit Card|          Paid|           YES|      1|
|       319|      19|       7900|  Debit Card|          Paid|           YES|      1|
|       311|      11|       8000|         UPI|       Pending|           YES|      1|
+----------+--------+-----------+------------+--------------+--------------+-------+

